# Metaclasses

## Singleton Pattern Implementation

The singleton pattern involves always returning the same instance when client code requests a new instance of a class.

Create a metaclass `Singleton` that implements this pattern. Note that a metaclass can be applied to multiple classes.

A metaclass can override several methods. Here, the simplest will be to redefine `__call__`.

In [ ]:
# Your code here

### Solution

In [ ]:
class SingletonMeta(type):

  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonMeta._instances[cls]


class A(metaclass=SingletonMeta):
    pass


a1 = A()
a2 = A()
print(a1 is a2)

## Modifying Methods

Write a metaclass that transforms all methods containing the word "compute" of a class with the following decorator, which logs calls to the methods it decorates:

In [ ]:
import functools


def log_calls(f):

  @functools.wraps(f)
  def wrapper(self, *args, **kwargs):
    result = f(self, *args, **kwargs)
    print(f"{f.__name__} called with args {args} and kwargs {kwargs} "
          f"computed {result}")
    return result

  return wrapper

In [ ]:
# Your code here

### Solution

In [ ]:

class LogComputeMeta(type):
  def __new__(self, name, bases, dct):
    new_dct = {k: (log_calls(v) if "compute" in k else v) for k, v in dct.items()}
    return super().__new__(self, name, bases, new_dct)

class A(metaclass=LogComputeMeta):
  def compute_addition(self, a, b):
    return a + b

a = A()
a.compute_addition(1, 2)

## Modifying Class Hierarchy

It is also possible with metaclasses to modify the parents of a class.

Create a metaclass that adds the `DataSaver` class to the parents of the produced class if it contains a `data` attribute.

In [ ]:
class DataSaver:
  def save(self):
    print(f"Saving data {self.data}…")


# Your code here

### Solution

In [ ]:
class DataSaverMeta(type):
  def __new__(self, name, bases, dct):
    if "data" in dct:
      bases += (DataSaver,)
    return super().__new__(self, name, bases, dct)


class A(metaclass=DataSaverMeta):
  data = [1, 2]


class B(metaclass=DataSaverMeta):
  a = [1, 2]


a = A()
b = B()
a.save()
try:
    b.save()
except AttributeError as error:
    print("B has no data attribute, so DataSaver is not among its parents:", error)

## Validating Subclasses: Metaclass or `__init_subclass__`

We want every subclass of `Exporter` to define an `extension` class attribute, a string starting with a dot, and a `TypeError` to be raised as soon as the class is defined otherwise.

1. Implement this check with an `ExporterMeta` metaclass.
2. Then implement it without a metaclass, with `__init_subclass__`.

In both cases, `class CsvExporter(Exporter): extension = ".csv"` must be accepted, and `class BrokenExporter(Exporter): pass` must raise a `TypeError`.

In [ ]:
# Your code here

### Solution

In [ ]:
class ExporterMeta(type):
    def __init__(cls, name, bases, namespace):
        super().__init__(name, bases, namespace)
        if bases and not str(getattr(cls, "extension", "")).startswith("."):
            raise TypeError(f"{cls.__name__} must define an extension starting with a dot")


class Exporter(metaclass=ExporterMeta):
    pass


class CsvExporter(Exporter):
    extension = ".csv"


try:

    class BrokenExporter(Exporter):
        pass

except TypeError as error:
    print(error)


# Without a metaclass
class Exporter:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if not str(getattr(cls, "extension", "")).startswith("."):
            raise TypeError(f"{cls.__name__} must define an extension starting with a dot")


class CsvExporter(Exporter):
    extension = ".csv"


try:

    class BrokenExporter(Exporter):
        pass

except TypeError as error:
    print(error)

`__init_subclass__` gives the same result without a metaclass: the code is simpler, and the class can still be combined with other classes that have their own metaclass.